In [0]:
from pyspark.sql import SparkSession

# สร้าง SparkSession
spark = SparkSession.builder \
    .appName("DataEng Workshop") \
    .getOrCreate()

# ทดสอบว่าทำงาน
print("Spark version:", spark.version)

Spark version: 4.1.0


In [0]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .load("/Workspace/Users/ekarat@as.nida.ac.th/Drafts/sample_data.csv")
df.show(5)

+--------------+------+-----------+-----+---+
|          name|amount|   category|price|qty|
+--------------+------+-----------+-----+---+
|      Notebook|  1200|Electronics| 1200|  1|
|Wireless Mouse|   450|Electronics|  150|  3|
|    Desk Chair|  2500|  Furniture| 2500|  1|
|    Coffee Mug|   320|    Kitchen|   80|  4|
|  Water Bottle|  1050|    Kitchen|  350|  3|
+--------------+------+-----------+-----+---+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import sum, avg

# เลือก column
df.select("name", "amount").show()
# กรองข้อมูล
df.filter(df["amount"] > 1000).show()
# Group by
df.groupBy("category").agg(
      sum("amount").alias("total"),
      avg("amount").alias("avg")
  ).show()
#เพิ่ม column ใหม่
df.withColumn("total", df["price"] * df["qty"]).show()

+--------------+------+
|          name|amount|
+--------------+------+
|      Notebook|  1200|
|Wireless Mouse|   450|
|    Desk Chair|  2500|
|    Coffee Mug|   320|
|  Water Bottle|  1050|
|     Desk Lamp|   850|
+--------------+------+

+------------+------+-----------+-----+---+
|        name|amount|   category|price|qty|
+------------+------+-----------+-----+---+
|    Notebook|  1200|Electronics| 1200|  1|
|  Desk Chair|  2500|  Furniture| 2500|  1|
|Water Bottle|  1050|    Kitchen|  350|  3|
+------------+------+-----------+-----+---+

+-----------+-----+------+
|   category|total|   avg|
+-----------+-----+------+
|    Kitchen| 1370| 685.0|
|Electronics| 1650| 825.0|
|  Furniture| 3350|1675.0|
+-----------+-----+------+

+--------------+------+-----------+-----+---+-----+
|          name|amount|   category|price|qty|total|
+--------------+------+-----------+-----+---+-----+
|      Notebook|  1200|Electronics| 1200|  1| 1200|
|Wireless Mouse|   450|Electronics|  150|  3|  450|


In [0]:
df.createOrReplaceTempView("sales") # สร้าง temp view
result = spark.sql("""
    SELECT category, SUM(amount) as total, COUNT(*) as count
    FROM sales
    WHERE amount > 0
    GROUP BY category
    ORDER BY total DESC
""")
result.show()

+-----------+-----+-----+
|   category|total|count|
+-----------+-----+-----+
|  Furniture| 3350|    2|
|Electronics| 1650|    2|
|    Kitchen| 1370|    2|
+-----------+-----+-----+



RDD operations are not supported on Serverless compute. Serverless uses Spark Connect, which has architectural restrictions that prevent RDD access.

In [0]:
# Use DataFrame operations directly instead of RDD
display(df.limit(5))

name,amount,category,price,qty
Notebook,1200,Electronics,1200,1
Wireless Mouse,450,Electronics,150,3
Desk Chair,2500,Furniture,2500,1
Coffee Mug,320,Kitchen,80,4
Water Bottle,1050,Kitchen,350,3
